# Qwen2.5-0.5B Q4_K_M · LongBench · Threshold T · Vivado Golden Data (v4)

**Mục tiêu:** (1) ghi **T thật theo từng sample × layer × head × mức sparsity**, (2) chấm LongBench, (3) xuất Q/K INT8 + `norm_sq`/proxy/T/KEEP mask cho Vivado. Bản này dùng **thư mục mới V4**, **không tái sử dụng** điểm LongBench v3, **không ghi đè** dữ liệu v2/v3.

**Luồng:** GGUF → Q/K sau RoPE (GQA K lặp đúng head) → lượng tử INT8 per head → sum-of-squares cho từng block 32 token → `proxy_uint64 = q_norm_sq × k_norm_sq` → T cho 50/70/80/90 → kiểm tra `proxy >= T` → block-sparse exact attention trên Q/K float → LongBench. Decode dùng eager dense.

**Tương thích RTL files:** `Norm_Base_Top.sv` gọi `Norm_Engine` nhưng file `Norm_Engine.sv` **chưa được cung cấp**. Các module đã có: `Proxy_Engine.sv`, `Threshold_Comparator.sv`, `Norm_RAM.sv`, `Causal_Block_Mask.sv`, `Block_FIFO.sv`, `Block_Scheduler.sv`. File HEX dùng **32 byte / mỗi bus 256-bit**; q/k 8-bit signed; norm 32-bit unsigned, proxy/T 64-bit unsigned, ID 8-bit.

**Lưu ý về thuật toán:** Giữ logic **bảo vệ các block đường chéo** của v3, chọn thêm Top-K các block ngoài đường chéo. Để khớp chính xác với `Threshold_Comparator.sv` (`proxy >= threshold_T`), đầu vào `threshold_T` **theo từng cặp pair** là 0 cho đường chéo và T cho các pair khác. `threshold_XX.hex` chứa **một ngưỡng T ngoài đường chéo** của mỗi sample/layer/head; `threshold_inputs_XX.hex` chứa ngưỡng thực đưa vào RTL theo `pairs.hex`. Nếu proxy bằng T, giữ tất cả block bằng T; vì vậy sparsity có thể chênh nhẹ so với target. Khác v2 thuần TopK và được công bố tường minh. Causal mask mức token vẫn được áp dụng trong diagonal block.

**Thực thi:** Google Colab → GPU T4 → Runtime → Run all. Drive sẽ có `results/threshold_trace_*.jsonl`, `results/threshold_summary.csv`, `results/longbench_scores.csv`, `vivado_testdata/`, checkpoint theo mẫu. Chạy lại sau ngắt có thể resume. `transformers` GGUF loader **giải lượng tử weight khi load**: benchmark không phải hiệu năng llama.cpp chạy Q4_K_M trực tiếp. Không thể kiểm thử GPU/full LongBench trong môi trường tạo notebook, nên đã tích hợp self-test và kiểm tra nhất quán output.


In [ ]:
# ============================================================
# 1. GOOGLE DRIVE + CONFIG
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter
import os, sys, json, math, gc, shutil, subprocess, time
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Cần GPU trong Colab: Runtime > Change runtime type > T4 GPU.')

# New result directory: never merge old v2 results with fixed v3 results.
ROOT = Path('/content/drive/MyDrive/QWEN_NORM_PROXY_Q4KM_RUN_V4_T_RTL')
PRED_ROOT = ROOT / 'predictions'
TRACE_ROOT = ROOT / 'threshold_traces'
TB_ROOT = ROOT / 'vivado_testdata'
RESULT_ROOT = ROOT / 'results'
CACHE_ROOT = ROOT / 'hf_cache'
for p in (ROOT, PRED_ROOT, RESULT_ROOT, CACHE_ROOT, TRACE_ROOT, TB_ROOT):
    p.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct-GGUF'
MODEL_BASE_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # tokenizer fallback only
GGUF_FILE = 'qwen2.5-0.5b-instruct-q4_k_m.gguf'
SEQ_LEN = 2048
TASKS = ['narrativeqa', 'qasper', 'gov_report']
SAMPLES_PER_TASK = 15
BLOCK_ROWS = 32
BLOCK_COLS = 32
LEVELS = [0, 50, 70, 80, 90]
FORCE_RERUN = False  # True: xóa checkpoints của V4 và tính lại toàn bộ
RUN_BENCHMARK = True  # False: chỉ chạy self-test và xuất Test A deterministic
EXPORT_TB_DATA = True
TB_SAMPLE_IDS = ['narrativeqa_000', 'qasper_000', 'gov_report_000']
TB_LAYER_INDICES = None  # tự chọn: đầu, giữa, cuối (sau khi load model)
TB_HEAD_INDICES = None   # tự chọn: head 0 và head giữa
KEEP_DIAGONAL = True  # v3 protected diagonal; drive RTL per-pair threshold=0 on diagonal
LONG_BENCH_ROOT = Path('/content/LongBench')
MODEL_LOCAL_DIR = Path('/content/qwen_q4km_local')
GGUF_CACHE_FILE = CACHE_ROOT / GGUF_FILE

# An optional, verified source file from v2 to avoid downloading twice.
OLD_VALID_FILE = Path('/content/drive/MyDrive/QWEN_NORM_PROXY_Q4KM_RUN_V2/hf_cache/qwen_q4km') / GGUF_FILE
V3_VALID_FILE = Path('/content/drive/MyDrive/QWEN_NORM_PROXY_Q4KM_RUN_V3_FIXED/hf_cache') / GGUF_FILE

print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
print('Results:', ROOT)
print('Model:', MODEL_ID)


In [ ]:
# ============================================================
# 2. INSTALL COMPATIBLE DEPENDENCIES + CLONE LONGBENCH v1
# ============================================================
# Gradio/diffusers from some new Colab images require huggingface-hub 1.x,
# but Transformers 4.57.x requires hub <1. They are UNUSED here.
# Uninstall only in the current disposable Colab runtime.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'gradio', 'gradio-client', 'diffusers'],
               check=False, stdout=subprocess.DEVNULL)

packages = [
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'datasets>=3.6.0,<4',
    'accelerate>=1.0,<2',
    'sentencepiece', 'gguf', 'rouge==1.0.1',
    'fuzzywuzzy', 'python-Levenshtein', 'jieba',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

from importlib import metadata as im
assert im.version('transformers') == '4.57.6', im.version('transformers')
assert im.version('huggingface-hub') == '0.36.2', im.version('huggingface-hub')
if 'transformers' in sys.modules and sys.modules['transformers'].__version__ != '4.57.6':
    raise RuntimeError('Transformers cũ vẫn nằm trong RAM. Restart session rồi Run all.')

if not (LONG_BENCH_ROOT / 'LongBench' / 'eval.py').exists():
    if LONG_BENCH_ROOT.exists():
        shutil.rmtree(LONG_BENCH_ROOT)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/THUDM/LongBench.git',
                    str(LONG_BENCH_ROOT)], check=True)

print('transformers:', im.version('transformers'))
print('huggingface-hub:', im.version('huggingface-hub'))
print('datasets:', im.version('datasets'))
print('LongBench evaluator:', LONG_BENCH_ROOT / 'LongBench' / 'eval.py')


In [ ]:
# ============================================================
# 3. VALIDATE / RESTORE / DOWNLOAD EXACT GGUF
# ============================================================
from huggingface_hub import hf_hub_download

MIN_GGUF_BYTES = 300 * 1024 * 1024  # Qwen2.5-0.5B Q4_K_M is >300 MiB

def check_gguf(path):
    path = Path(path)
    if not path.is_file():
        return False
    if path.stat().st_size < MIN_GGUF_BYTES:
        return False
    with path.open('rb') as f:
        return f.read(4) == b'GGUF'

MODEL_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
gguf_path = MODEL_LOCAL_DIR / GGUF_FILE

if check_gguf(gguf_path):
    print('Reuse valid local file:', gguf_path)
else:
    # Reuse a verified Drive copy. NEVER use an unvalidated HF snapshots file.
    candidates = [GGUF_CACHE_FILE, V3_VALID_FILE, OLD_VALID_FILE]
    verified = next((p for p in candidates if check_gguf(p)), None)
    if verified is not None:
        print('Restore verified GGUF from Drive:', verified)
        shutil.copy2(verified, gguf_path)
    else:
        # Clean the invalid local file and force download when needed.
        if gguf_path.exists():
            gguf_path.unlink()
        print('Downloading verified Q4_K_M GGUF from Hugging Face...')
        downloaded = hf_hub_download(
            repo_id=MODEL_ID,
            filename=GGUF_FILE,
            local_dir=str(MODEL_LOCAL_DIR),
            force_download=True,
        )
        gguf_path = Path(downloaded)

if not check_gguf(gguf_path):
    first = gguf_path.read_bytes()[:16] if gguf_path.is_file() else b''
    raise ValueError(f'File tải về KHÔNG phải GGUF: {gguf_path}. Header={first!r}')

# Save a verified copy for next Colab session, without touching existing valid copy.
if not check_gguf(GGUF_CACHE_FILE):
    tmp = GGUF_CACHE_FILE.with_suffix('.part')
    shutil.copy2(gguf_path, tmp)
    if not check_gguf(tmp):
        tmp.unlink(missing_ok=True)
        raise RuntimeError('Lỗi khi sao chép GGUF vào Drive.')
    os.replace(tmp, GGUF_CACHE_FILE)

print('GGUF LOCAL PATH:', gguf_path)
print('GGUF FIRST 4 BYTES:', gguf_path.open('rb').read(4))
print('GGUF SIZE MiB:', round(gguf_path.stat().st_size / (1024**2), 2))
print('Valid GGUF: PASS')


In [ ]:
# ============================================================
# 4. LOAD GGUF FROM THE VERIFIED LOCAL DIRECTORY (NO SNAPSHOTS)
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load both from the same local directory. The old notebook downloaded to a
# local_dir but loaded again from MODEL_ID / stale snapshots cache.
try:
    tokenizer = AutoTokenizer.from_pretrained(
        str(MODEL_LOCAL_DIR), gguf_file=GGUF_FILE, use_fast=True,
    )
    tokenizer_source = 'GGUF metadata'
except Exception as exc:
    print('GGUF tokenizer metadata not supported; use the official Qwen tokenizer:', repr(exc))
    tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE_ID, use_fast=True)
    tokenizer_source = MODEL_BASE_ID

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_LOCAL_DIR),
    gguf_file=GGUF_FILE,
    dtype=torch.float16,
    device_map='auto',
    attn_implementation='eager',
    low_cpu_mem_usage=True,
)
model.eval()
if model.config._attn_implementation != 'eager':
    raise RuntimeError('Phải dùng eager attention cho NormProxy patch.')

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

device = model.get_input_embeddings().weight.device
assert device.type == 'cuda', f'Model không nằm trên GPU: {device}'

print('Model loaded from verified GGUF:', gguf_path)
print('Tokenizer source:', tokenizer_source)
print('Model class:', type(model).__name__)
print('dtype:', next(model.parameters()).dtype)
print('device:', device)
print('layers:', model.config.num_hidden_layers)
print('Q heads:', model.config.num_attention_heads)
print('KV heads:', model.config.num_key_value_heads)
print('head_dim:', model.config.hidden_size // model.config.num_attention_heads)


In [ ]:
# 5. RUNTIME CONFIG VALIDATION + OUTPUT RULES
TB_LAYER_INDICES = (TB_LAYER_INDICES if TB_LAYER_INDICES is not None else sorted({0, model.config.num_hidden_layers // 2, model.config.num_hidden_layers - 1}))
TB_HEAD_INDICES = (TB_HEAD_INDICES if TB_HEAD_INDICES is not None else sorted({0, model.config.num_attention_heads // 2}))
assert BLOCK_ROWS == BLOCK_COLS == 32
assert all(0 <= i < model.config.num_hidden_layers for i in TB_LAYER_INDICES)
assert all(0 <= i < model.config.num_attention_heads for i in TB_HEAD_INDICES)
assert model.config.num_attention_heads % model.config.num_key_value_heads == 0
head_dim = getattr(model.config, 'head_dim', model.config.hidden_size // model.config.num_attention_heads)
MAX_NORM_SQ = BLOCK_ROWS * head_dim * (127**2)
assert MAX_NORM_SQ <= 2**32-1, f'NORM_W=32 would overflow: {MAX_NORM_SQ}'
assert MAX_NORM_SQ**2 <= 2**64-1, 'SCORE_W=64 would overflow'
assert (SEQ_LEN + 31) // 32 < 256, 'ID_W=8 overflow'
print('TB layer indices:', TB_LAYER_INDICES, '| head indices:', TB_HEAD_INDICES)
print('Head dim:',head_dim,'| max possible norm_sq:',MAX_NORM_SQ)
print('Threshold policy: protect diagonal + threshold for off-diagonal; each pair uses threshold=0 if diagonal else T')


In [ ]:
# ============================================================
# 5. LOAD FIXED LONGBENCH SAMPLE SET (15/TASK)
# ============================================================
CONFIG_DIR = LONG_BENCH_ROOT / 'LongBench' / 'config'
with (CONFIG_DIR / 'dataset2prompt.json').open(encoding='utf-8') as f:
    dataset2prompt = json.load(f)
with (CONFIG_DIR / 'dataset2maxlen.json').open(encoding='utf-8') as f:
    dataset2maxlen = json.load(f)

from datasets import load_dataset

samples = []
dataset_sources = {}
for task in TASKS:
    try:
        ds = load_dataset('THUDM/LongBench', task, split='test')
        dataset_sources[task] = 'THUDM/LongBench'
    except Exception as exc:
        # Modern datasets installations sometimes reject script-based repos;
        # public Parquet mirror is a format conversion of LongBench v1.
        print(f'Official HF loader unavailable for {task}: {type(exc).__name__}: {exc}')
        print('Using format-only LongBench Parquet mirror for', task)
        ds = load_dataset('VisionTheta/LongBench-Parquet', task, split='test')
        dataset_sources[task] = 'VisionTheta/LongBench-Parquet'

    if len(ds) < SAMPLES_PER_TASK:
        raise RuntimeError(f'{task} has only {len(ds)} samples')
    for idx, item in enumerate(ds.select(range(SAMPLES_PER_TASK))):
        samples.append({
            'sample_id': f'{task}_{idx:03d}',
            'task': task,
            'context': item['context'],
            'input': item['input'],
            'answers': item['answers'],
            'all_classes': item.get('all_classes') or [],
            'length': item.get('length'),
        })

assert len(samples) == SAMPLES_PER_TASK * len(TASKS)
print('TOTAL:', len(samples))
print('BY TASK:', Counter(s['task'] for s in samples))
print('SOURCES:', dataset_sources)

assert set(TB_SAMPLE_IDS).issubset({s['sample_id'] for s in samples}), 'TB_SAMPLE_IDS not present in loaded benchmark sample set'


## Bước 7 · Golden model dùng chung cho software và RTL

- `q_int8.hex`/`k_int8.hex`: 64 hex digits/line = **256-bit beat**, lane 0 là byte thấp nhất `[7:0]`, lane 31 `[255:248]`; mỗi block đi theo thứ tự token-major rồi head dimension, zero-pad beat cuối nếu cần.
- Một `Norm_Engine` job nhận một block ID (32 tokens × head_dim), dữ liệu tương ứng là các dòng liên tiếp `beats_per_block` trong HEX; `last` trên beat cuối. **Cần xác nhận nội bộ Norm_Engine** khi có source.
- `q_norm_sq.hex`, `k_norm_sq.hex`: 8 hex digits cho unsigned 32-bit; `proxy.hex`, `threshold_XX.hex`: 16 hex digits cho unsigned 64-bit. `pairs.hex`: 4 hex digits `{q_id[7:0], k_id[7:0]}`, các dòng theo thứ tự causal `(0,0),(1,0),(1,1),...`.
- `expected_mask_XX.hex`: `0`/`1` cho từng cặp theo `pairs.hex`; `threshold_XX.hex`: **T 64-bit cho các pair ngoài đường chéo**; `threshold_inputs_XX.hex`: **T per-pair thực sự nối vào cổng `threshold_T`**, bằng 0 trên đường chéo và T ở ngoài đường chéo. Các ngưỡng ghi đúng uint64, không làm tròn float.


In [ ]:
# 7. GOLDEN INT8/NORM/PROXY/T/MASK — bit-accurate with provided RTL widths
import numpy as np
import csv, hashlib

LEVELS_SPARSE = [50, 70, 80, 90]
NORM_W, SCORE_W, ID_W, BUS_BYTES = 32, 64, 8, 32
THRESHOLD_POLICY = 'protected_diagonal_plus_topk_optional_per_pair_threshold_input'

def tensor_quant8(x):
    # Exactly the current experiment's per-head symmetric dynamic INT8; torch.round ties-to-even.
    # Scale and rounding in FP32 for deterministic signed INT8 conversion.
    x32=x.detach().float()
    scale = torch.clamp(x32.abs().amax() / 127., min=1e-8)
    values = torch.round(x32 / scale).clamp(-127,127).to(torch.int8)
    return values, float(scale.item())

def quantize_pair(q,k):
    q8,qs = tensor_quant8(q)
    k8,ks = tensor_quant8(k)
    return q8,k8,qs,ks

def compute_golden_from_int8(q8,k8):
    assert q8.ndim == k8.ndim == 2 and q8.shape[1] == k8.shape[1]
    nq = (q8.shape[0]+BLOCK_ROWS-1)//BLOCK_ROWS
    nk = (k8.shape[0]+BLOCK_COLS-1)//BLOCK_COLS
    def sqblocks(x, block):
        out=[]
        for a in range(0,x.shape[0],block):
            v=x[a:a+block].to(torch.int64)
            out.append((v*v).sum())
        return torch.stack(out)
    qn = sqblocks(q8,BLOCK_ROWS)
    kn = sqblocks(k8,BLOCK_COLS)
    if bool((qn > 2**NORM_W-1).any()) or bool((kn > 2**NORM_W-1).any()):
        raise OverflowError('Norm Engine NORM_W=32 overflow')
    proxy = qn[:,None]*kn[None,:]
    if bool((proxy<0).any()):raise OverflowError('proxy signed int64 overflow')
    valid = torch.tril(torch.ones((nq,nk),device=q8.device,dtype=torch.bool))
    return qn,kn,proxy,valid

def threshold_result(proxy,valid,level):
    assert 0 < level < 100 and KEEP_DIAGONAL
    candidates = int(valid.sum().item())
    if candidates==0:raise ValueError('No causal candidate blocks')
    diag=torch.zeros_like(valid)
    n_diag=min(proxy.shape[0],proxy.shape[1]); diag_idx=torch.arange(n_diag,device=proxy.device)
    diag[diag_idx,diag_idx]=True
    optional=valid & ~diag
    values=proxy[optional]
    n_optional=int(values.numel())
    # Target rounded like original v3; diagonal ALWAYS kept for safe attention.
    requested_kept=max(0,min(candidates,round(candidates*(1.-level/100.))))
    budget=max(n_diag,requested_kept)
    n_remaining=max(0,min(n_optional,budget-n_diag))
    if n_optional==0:
        T=0
    elif n_remaining==0:
        T=int(values.max().item())+1  # 1 above max means NO optional pair kept
    elif n_remaining==n_optional:
        T=0
    else:
        T=int(torch.sort(values,descending=True,stable=True).values[n_remaining-1].item())
    if T>2**64-1:raise OverflowError('T > 64-bit RTL threshold capacity')
    # Norm_Base_Top captures threshold_T for EACH pair; TB driver can set 0 for diagonal,
    # T for off-diagonal without modifying the 64-bit >= comparator hardware.
    pair_T=torch.full_like(proxy,T)
    pair_T[diag]=0
    keep=valid & (proxy>=pair_T)
    assert bool(torch.diagonal(keep).all()),'diagonal MUST stay kept'
    n_keep=int(keep.sum().item())
    ties=int(((proxy==T)&optional).sum().item())
    return {
       'target_sparsity_pct':int(level),'requested_keep_count':requested_kept,
       'target_keep_count':budget,'candidate_count':candidates,
       'topk_T':T,'threshold_T':T,'diagonal_blocks':n_diag,
       'optional_candidate_count':n_optional,'optional_keep_budget':n_remaining,
       'ties_at_T':ties,'kept_count':n_keep,'pruned_count':candidates-n_keep,
       'actual_sparsity_pct':100.0*(candidates-n_keep)/candidates,
       'mask':keep,'threshold_input':pair_T,
    }

def golden_all_levels(q8,k8):
    qn,kn,proxy,valid=compute_golden_from_int8(q8,k8)
    levels={lev:threshold_result(proxy,valid,lev) for lev in LEVELS_SPARSE}
    return qn,kn,proxy,valid,levels

def assert_threshold_matches_rtl(g):
    # One pure unsigned >= per causal candidate; the comparator RTL has no tie-breaker.
    _,_,proxy,valid,levels=g
    for lev,obj in levels.items():
        t=obj['threshold_T']; per_pair_T=obj['threshold_input']
        predicted=valid & (proxy>=per_pair_T)
        assert torch.equal(predicted,obj['mask']),f'RTL comparator mismatch at {lev}%'
        assert int(predicted.sum().item())+obj['pruned_count']==obj['candidate_count']
        assert all(int(z)==0 for z in torch.diagonal(per_pair_T).tolist())
        assert 0<=t<=2**SCORE_W-1

def atomic_json(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False),encoding='utf-8')
    os.replace(tmp,path)

def atomic_text(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');tmp.write_text(value,encoding='utf-8')
    os.replace(tmp,path)

def atomic_jsonl(path,rows):
    atomic_text(path,''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows))

def mem_256_lines(x):
    arr = x.detach().cpu().contiguous().numpy().astype(np.int16).reshape(-1)
    block_elems=BLOCK_ROWS*x.shape[1] if x.ndim==2 else None
    assert block_elems and x.shape[0]>0
    # Always pad to the next 256-bit beat **at each block boundary**.
    beats_per_block=(block_elems+BUS_BYTES-1)//BUS_BYTES
    chunks=[]
    for token_start in range(0,x.shape[0],BLOCK_ROWS):
        one=x[token_start:token_start+BLOCK_ROWS].detach().cpu().numpy().astype(np.int16).reshape(-1)
        for i in range(0,len(one),BUS_BYTES):
            a=one[i:i+BUS_BYTES]
            packed=sum((int(v)&0xff)<<(8*j) for j,v in enumerate(a))
            chunks.append(f'{packed:064x}')
        # Partial final block contains fewer beats; pad unused beats with zeros.
        missing=beats_per_block-((len(one)+BUS_BYTES-1)//BUS_BYTES)
        chunks.extend(['0'*64]*missing)
    return chunks,beats_per_block

def save_hex(path,nums,width):
    assert all(0<=int(v)<2**(4*width) for v in nums),f'{path} out of bit range'
    atomic_text(path,''.join(f'{int(v):0{width}x}\n' for v in nums))

def export_case(case_name,q8,k8,qscale,kscale,extra):
    case_dir=TB_ROOT/case_name
    if (case_dir/'metadata.json').exists() and not FORCE_RERUN:
        return json.loads((case_dir/'metadata.json').read_text())
    qn,kn,proxy,valid,levels = golden_all_levels(q8,k8)
    assert_threshold_matches_rtl((qn,kn,proxy,valid,levels))
    qbeats,bq=mem_256_lines(q8);kbeats,bk=mem_256_lines(k8)
    save_hex(case_dir/'q_int8.hex',[int(x,16) for x in qbeats],64)
    save_hex(case_dir/'k_int8.hex',[int(x,16) for x in kbeats],64)
    qnlist=[int(x) for x in qn.tolist()];knlist=[int(x) for x in kn.tolist()]
    save_hex(case_dir/'q_norm_sq.hex',qnlist,8)
    save_hex(case_dir/'k_norm_sq.hex',knlist,8)
    pairs=torch.nonzero(valid,as_tuple=False).cpu().tolist()
    save_hex(case_dir/'pairs.hex',[(int(i)<<8)|int(j) for i,j in pairs],4)
    save_hex(case_dir/'expected_proxy.hex',[int(proxy[i,j]) for i,j in pairs],16)
    atomic_text(case_dir/'expected_proxy.txt',''.join(f'{i},{j},{int(proxy[i,j])}\n' for i,j in pairs))
    # Decimal copies ease debug/plot, HEX copies directly match Verilog $readmemh.
    atomic_text(case_dir/'q_norm_sq.txt','\n'.join(str(z) for z in qnlist)+'\n')
    atomic_text(case_dir/'k_norm_sq.txt','\n'.join(str(z) for z in knlist)+'\n')
    for lev,obj in levels.items():
        save_hex(case_dir/f'threshold_{lev}.hex',[obj['threshold_T']],16)
        atomic_text(case_dir/f'threshold_{lev}.txt',str(obj['threshold_T'])+'\n')
        save_hex(case_dir/f'threshold_inputs_{lev}.hex',[int(obj['threshold_input'][i,j]) for i,j in pairs],16)
        save_hex(case_dir/f'expected_mask_{lev}.hex',[int(obj['mask'][i,j]) for i,j in pairs],1)
        # full block matrix: causal-invalid entries are 0
        maskrows=[''.join(str(int(v)) for v in row) for row in obj['mask'].to(torch.int8).cpu().tolist()]
        atomic_text(case_dir/f'expected_mask_matrix_{lev}.txt','\n'.join(maskrows)+'\n')
    m={
       'case_id':case_name,'model':MODEL_ID,'checkpoint':GGUF_FILE,
       'source':extra,'q_len':int(q8.shape[0]),'k_len':int(k8.shape[0]),
       'head_dim':int(q8.shape[1]),'q_blocks':int(qn.numel()),'k_blocks':int(kn.numel()),
       'block_tokens':BLOCK_ROWS,'q_beats_per_block':bq,'k_beats_per_block':bk,
       'bus_width_bits':256,'int8_lane_order':'lane0 is bits[7:0] / lane31 is bits[255:248]',
       'stream_order':'block_id ascending, row-major tokens × head_dim, zero-pad each block to fixed beats_per_block',
       'quantization':'symmetric per-head dynamic int8; FP32 max_abs/127 clamped >=1e-8; FP32 torch.round ties-to-even',
       'q_scale':float(qscale),'k_scale':float(kscale),
       'norm_definition':'sum of squared signed INT8 values across entire 32-token × head_dim block',
       'proxy_definition':'uint64(q_norm_sq)*uint64(k_norm_sq)',
       'causal_valid':'k_block_id <= q_block_id',
       'threshold_rule':THRESHOLD_POLICY,'threshold_values':{str(lev):{k:v for k,v in obj.items() if k not in ('mask','threshold_input')} for lev,obj in levels.items()},
       'pairs_order':'valid row-major k<=q, matches pairs.hex + expected_proxy.hex + expected_mask_XX.hex',
       'rtl_params':{'ID_W':ID_W,'NORM_W':NORM_W,'SCORE_W':SCORE_W},
       'missing_rtl_source':'Norm_Engine.sv (not included in supplied 7 SV files)',
    }
    atomic_json(case_dir/'metadata.json',m)
    return m

# Golden-model small deterministic unit tests on torch CPU/GPU; independent of Qwen.
qsmall=torch.tensor([[(-11*i+7*j)%39-19 for j in range(head_dim)] for i in range(64)],dtype=torch.int8)
ksmall=torch.tensor([[(13*i-3*j)%39-19 for j in range(head_dim)] for i in range(64)],dtype=torch.int8)
small_g=golden_all_levels(qsmall,ksmall)
assert_threshold_matches_rtl(small_g)
assert all(int(x) <= 2**32-1 for x in small_g[0])
if EXPORT_TB_DATA:
    toy_meta=export_case('test_deterministic_64',qsmall,ksmall,1.0,1.0,{'kind':'small_deterministic','seed':'analytic_formula','stage':'test_A'})
print('PASS: uint32 norm, uint64 proxy, threshold >=, hex packing + deterministic test A.')


## Bước 8 · Hook Qwen2 eager attention (sau RoPE, GQA), trace T theo mẫu

- Chạy Dense: **không đổi output attention** nhưng lấy một số activation thực tế xuất Vivado TB cho cả bốn sparsity trên **cùng Q/K**.
- Chạy Sparse: mỗi sample ghi `threshold_T` **ở từng layer và head**, cùng `topk_T`, target/actual sparsity, lượng tử scale và số block. Các trace viết theo sample vào Drive, có thể resume.
- **Không tạo threshold giả** khi Dense hoặc decode; cả hai đi thẳng eager gốc. Mask được tạo từ proxy số nguyên và so sánh `>=T`, tương đương `Threshold_Comparator.sv`.


In [ ]:
# 8. PATCH TRANSFORMERS 4.57.6 EAGER; no modification to Qwen2Attention.forward
import torch.nn.functional as F
import transformers.models.qwen2.modeling_qwen2 as qwen2_module
from transformers.models.qwen2.modeling_qwen2 import repeat_kv

assert hasattr(qwen2_module,'eager_attention_forward')
_ORIGINAL_EAGER = qwen2_module.eager_attention_forward
CURRENT_LEVEL = 0
ACTIVE_SAMPLE = None
ACTIVE_TRACE = []
STATS = {}
TB_CASE_IDS = set()

def reset_stats(level=0):
    global STATS
    STATS={'target_sparsity':int(level),'attention_calls':0,'candidate_blocks':0,
           'kept_blocks':0,'pruned_blocks':0,'causal_blocks':0,
           'proxy_computations':0,'protected_diagonal_blocks':0,
           'threshold_ties':0}

def capture_tb_once(module,q,k,b,h):
    if not EXPORT_TB_DATA or ACTIVE_SAMPLE not in TB_SAMPLE_IDS: return
    layer=int(getattr(module,'layer_idx',-1))
    if layer not in TB_LAYER_INDICES or h not in TB_HEAD_INDICES: return
    if q.shape[0]<BLOCK_ROWS or q.shape[0]!=k.shape[0]: return
    case_id=f'test_{ACTIVE_SAMPLE}_L{layer:02d}_H{h:02d}_B{b}'
    if case_id in TB_CASE_IDS: return
    TB_CASE_IDS.add(case_id)
    q8,k8,qs,ks=quantize_pair(q,k)
    m=export_case(case_id,q8,k8,qs,ks,{
       'kind':'real_qwen','stage':'dense_prefill_after_rope',
       'sample_id':ACTIVE_SAMPLE,'layer':layer,'attention_head':h,'batch':b,
       'gqa_note':'key repeated from grouped KV head to match query head',
    })
    print('VIVADO CASE:',case_id,'Q blocks',m['q_blocks'],'K blocks',m['k_blocks'])

def normproxy_eager(module,query,key,value,attention_mask,scaling,dropout=0.0,**kwargs):
    global STATS,ACTIVE_TRACE
    bsz,num_heads,tq,dim=query.shape
    tk=int(key.shape[-2])
    # Prefill only. No sparse interference with incremental cache/decode.
    if tq < BLOCK_ROWS or tq!=tk:
        return _ORIGINAL_EAGER(module,query,key,value,attention_mask,
                               scaling=scaling,dropout=dropout,**kwargs)
    kr=repeat_kv(key,module.num_key_value_groups)
    if CURRENT_LEVEL == 0:
        if ACTIVE_SAMPLE in TB_SAMPLE_IDS and EXPORT_TB_DATA:
            for b in range(bsz):
                for h in TB_HEAD_INDICES:
                    capture_tb_once(module,query[b,h],kr[b,h],b,h)
        return _ORIGINAL_EAGER(module,query,key,value,attention_mask,
                               scaling=scaling,dropout=dropout,**kwargs)
    vr=repeat_kv(value,module.num_key_value_groups)
    scores=torch.full((bsz,num_heads,tq,tk),float('-inf'),device=query.device,dtype=query.dtype)
    layer=int(getattr(module,'layer_idx',-1))
    for b in range(bsz):
        for h in range(num_heads):
            # Detached INT8 only controls selection; exact float attention for kept blocks.
            with torch.no_grad():
                q8,k8,qs,ks=quantize_pair(query[b,h],kr[b,h])
                qn,kn,proxy,valid=compute_golden_from_int8(q8,k8)
                r=threshold_result(proxy,valid,CURRENT_LEVEL)
                keep=r['mask']
                assert_threshold_matches_rtl((qn,kn,proxy,valid,{CURRENT_LEVEL:r}))
            candidate=r['candidate_count'];kept=r['kept_count']
            STATS['attention_calls']+=1
            STATS['candidate_blocks']+=candidate
            STATS['kept_blocks']+=kept
            STATS['pruned_blocks']+=candidate-kept
            STATS['causal_blocks']+=int(valid.numel())-candidate
            STATS['proxy_computations']+=candidate
            STATS['protected_diagonal_blocks']+=r['diagonal_blocks']
            STATS['threshold_ties']+=r['ties_at_T']
            ACTIVE_TRACE.append({
                'sample_id':ACTIVE_SAMPLE,'batch':b,'layer':layer,'head':h,
                'q_len':tq,'k_len':tk,'q_blocks':int(qn.numel()),'k_blocks':int(kn.numel()),
                'q_scale':qs,'k_scale':ks,
                'target_sparsity_pct':CURRENT_LEVEL,'target_keep_count':r['target_keep_count'],
                'candidate_count':candidate,'topk_T':r['topk_T'],'threshold_T':r['threshold_T'],
                'diagonal_blocks':r['diagonal_blocks'],'ties_at_T':r['ties_at_T'],
                'kept_count':kept,'pruned_count':candidate-kept,
                'actual_sparsity_pct':r['actual_sparsity_pct'],
            })
            # Scores computed **only** for kept blocks; true FP16 QK products.
            for i in range(keep.shape[0]):
                for j in torch.nonzero(keep[i],as_tuple=False).flatten().tolist():
                    r0,r1=i*BLOCK_ROWS,min((i+1)*BLOCK_ROWS,tq)
                    c0,c1=j*BLOCK_COLS,min((j+1)*BLOCK_COLS,tk)
                    scores[b,h,r0:r1,c0:c1]=(query[b,h,r0:r1] @ kr[b,h,c0:c1].transpose(0,1))*scaling
    # Causal semantics also hold INSIDE diagonal blocks.
    future = torch.arange(tk,device=query.device)[None,:]>torch.arange(tq,device=query.device)[:,None]
    scores.masked_fill_(future[None,None],float('-inf'))
    if attention_mask is not None:
        scores=scores+attention_mask[:,:,:,:tk]
    prob=F.softmax(scores,dim=-1,dtype=torch.float32).to(query.dtype)
    if not torch.isfinite(prob).all():
        raise FloatingPointError('Sparse softmax produced NaN/Inf. Check global T cap and masks.')
    prob=F.dropout(prob,p=dropout,training=module.training)
    return torch.matmul(prob,vr).transpose(1,2).contiguous(),None

def install_level(level):
    global CURRENT_LEVEL
    assert level in LEVELS
    CURRENT_LEVEL=int(level)
    reset_stats(level)
    qwen2_module.eager_attention_forward=normproxy_eager
    print('Installed eager hook for level:',level,'(dense capture)' if level==0 else '(sparse threshold trace)')

def restore_dense():
    global CURRENT_LEVEL
    CURRENT_LEVEL=0
    qwen2_module.eager_attention_forward=_ORIGINAL_EAGER

restore_dense()
reset_stats()
print('PASS: eager hook ready; original function retained for restoration.')


In [ ]:
# 9. QUICK SELF-TEST — model forward, trace T, decode and exact RTL >= comparator
probe=('Long-context block sparse attention is evaluated using a deterministic test. '*12)
probe_inputs=tokenizer(probe,return_tensors='pt').to(device)
assert int(probe_inputs['input_ids'].shape[1])>=BLOCK_ROWS
install_level(50)
ACTIVE_SAMPLE='selfcheck'
ACTIVE_TRACE=[]
with torch.inference_mode():
    res=model(**probe_inputs,use_cache=True)
assert torch.isfinite(res.logits).all()
assert STATS['attention_calls']>0 and len(ACTIVE_TRACE)>0
assert all(r['threshold_T']>=0 for r in ACTIVE_TRACE)
with torch.inference_mode():
    decoded=model.generate(**probe_inputs,max_new_tokens=2,do_sample=False,use_cache=True,pad_token_id=tokenizer.pad_token_id)
assert decoded.shape[1]>probe_inputs['input_ids'].shape[1]
print('PASS Qwen sparse prefill + T trace + dense decode; example T:',ACTIVE_TRACE[0]['threshold_T'])
restore_dense();ACTIVE_SAMPLE=None;ACTIVE_TRACE=[]


In [ ]:
# ============================================================
# 8. LONGBENCH PROMPTS + EVALUATION HELPERS
# ============================================================
def build_prompt(sample):
    return dataset2prompt[sample['task']].format(**sample)

def generate_answer(sample):
    prompt = build_prompt(sample)
    inputs = tokenizer(prompt,return_tensors='pt',truncation=True,
                       max_length=SEQ_LEN).to(device)
    max_new_tokens = int(dataset2maxlen.get(sample['task'],64))
    with torch.inference_mode():
        out = model.generate(**inputs,max_new_tokens=max_new_tokens,
                             do_sample=False,use_cache=True,
                             pad_token_id=tokenizer.pad_token_id)
    generated = out[0,inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated,skip_special_tokens=True)

def level_name(level):
    return 'dense_q4km' if level == 0 else f'qwen_q4km_norm_{level}pct_32x32'

def atomic_write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_name(path.name+'.tmp')
    with temp.open('w',encoding='utf-8') as f:
        json.dump(obj,f,ensure_ascii=False,indent=2)
    os.replace(temp,path)

def save_predictions(level,records):
    dst = PRED_ROOT / level_name(level)
    dst.mkdir(parents=True,exist_ok=True)
    for task in TASKS:
        rows = [r for r in records if r['task']==task]
        target = dst / f'{task}.jsonl'
        temp = target.with_suffix('.jsonl.tmp')
        with temp.open('w',encoding='utf-8') as f:
            for row in rows:
                f.write(json.dumps(row,ensure_ascii=False)+'\n')
        os.replace(temp,target)
    atomic_write_json(dst/'manifest.json',{
        'model':MODEL_ID,'gguf_file':GGUF_FILE,'block_size':[BLOCK_ROWS,BLOCK_COLS],
        'level':level,'samples':len(records),'tasks':TASKS,
        'dataset_sources':dataset_sources,'seq_len':SEQ_LEN,
        'proxy':'P=N_Q*N_K; N=sum(int8**2)',
        'threshold_policy':THRESHOLD_POLICY,
        'note':'GGUF weights dequantized by Transformers; T is emitted separately for each attention head; not llama.cpp Q4_K_M runtime',
    })
    return dst

def prepare_longbench_pred(level):
    name = level_name(level)
    src = PRED_ROOT / name
    dst = LONG_BENCH_ROOT / 'LongBench' / 'pred' / name
    # Evaluator paths are resolved RELATIVE to LongBench/LongBench, not repository root.
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True,exist_ok=True)
    shutil.copytree(src,dst)
    for task in TASKS:
        if not (dst/f'{task}.jsonl').exists():
            raise FileNotFoundError(dst/f'{task}.jsonl')
    return name,dst

def run_longbench_eval(level):
    name,dst = prepare_longbench_pred(level)
    runner_dir = LONG_BENCH_ROOT / 'LongBench'
    cmd = [sys.executable,str(runner_dir/'eval.py'),'--model',name]
    cp = subprocess.run(cmd,cwd=str(runner_dir),text=True,capture_output=True)
    if cp.stdout:
        print(cp.stdout)
    if cp.stderr:
        print(cp.stderr)
    if cp.returncode != 0:
        raise RuntimeError(f'LongBench eval failed for {name}: code {cp.returncode}')
    score_file = dst/'result.json'
    if not score_file.exists():
        raise FileNotFoundError(score_file)
    return json.loads(score_file.read_text(encoding='utf-8'))


## Bước 11 · Checkpoint per sample, benchmark và báo cáo T

Mỗi mức sparsity lưu trace của từng mẫu ngay khi mẫu hoàn tất. **Kết quả một mức chỉ được đánh dấu DONE nếu đủ trace của tất cả các sample**, có predictions và result.json. Sau khi benchmark hoàn thành, một cell tổng hợp tạo `threshold_trace_LEVEL.jsonl` (đọc được từng dòng), `threshold_summary.csv`, `threshold_distribution.csv`, `sparsity_summary.csv`, `longbench_scores.csv`, `execution_stats.json`. Quan trọng: tất cả đường dẫn đều nằm trên Drive.


In [ ]:
# 11. RUN A LEVEL WITH PER-SAMPLE TRACE + RESUME
# Import LongBench evaluator helpers from section 10 (already defined above).

def trace_sample_path(level,sample_id):
    return TRACE_ROOT/level_name(level)/f'{sample_id}.jsonl'

def run_level_v4(level):
    assert level in LEVELS
    name=level_name(level)
    target=RESULT_ROOT/f'{name}.json'
    pred_dir=PRED_ROOT/name
    checkpoint_file=PRED_ROOT/f'{name}__checkpoint/checkpoint.json'
    level_trace_dir=TRACE_ROOT/name
    if not FORCE_RERUN and target.exists() and all((pred_dir/f'{t}.jsonl').exists() for t in TASKS) and (
       level==0 or all(trace_sample_path(level,s['sample_id']).exists() for s in samples)):
        print('REUSE COMPLETE V4 LEVEL:',target)
        return json.loads(target.read_text(encoding='utf-8'))
    if FORCE_RERUN:
        for d in [pred_dir,checkpoint_file.parent,level_trace_dir]:
            if d.exists():shutil.rmtree(d)
        target.unlink(missing_ok=True)
    install_level(level)  # dense still hooks so reference activations export to Vivado
    records=[]
    if checkpoint_file.exists() and not FORCE_RERUN:
        cp=json.loads(checkpoint_file.read_text(encoding='utf-8'))
        records=cp.get('records',[])
        expected_ids=[s['sample_id'] for s in samples[:len(records)]]
        if [r['sample_id'] for r in records]!=expected_ids or cp.get('level')!=level:
            raise RuntimeError('Checkpoint mismatched sample order/level: '+str(checkpoint_file))
        if level!=0:
            for r in records:
                if not trace_sample_path(level,r['sample_id']).exists():
                    raise RuntimeError('Missing threshold trace despite checkpoint: '+r['sample_id'])
        STATS.update(cp.get('stats',{}))
        print('RESUMING',name,len(records),'/',len(samples))
    start=time.time()
    for i in range(len(records),len(samples)):
        global ACTIVE_SAMPLE,ACTIVE_TRACE
        s=samples[i];ACTIVE_SAMPLE=s['sample_id'];ACTIVE_TRACE=[]
        print(f'[{i+1}/{len(samples)}] {name} / {ACTIVE_SAMPLE}',flush=True)
        prediction=generate_answer(s)
        if level>0:
            if not ACTIVE_TRACE:raise RuntimeError(f'No T captured for {ACTIVE_SAMPLE} at level {level}')
            atomic_jsonl(trace_sample_path(level,ACTIVE_SAMPLE),ACTIVE_TRACE)
        records.append({
            'sample_id':s['sample_id'],'pred':prediction,'answers':s['answers'],
            'task':s['task'],'all_classes':s['all_classes'],'length':s['length'],
        })
        atomic_json(checkpoint_file,{'level':level,'records':records,'stats':dict(STATS)})
        print('Saved sample',i+1,'T records:',len(ACTIVE_TRACE),'elapsed minutes:',round((time.time()-start)/60,1),flush=True)
        ACTIVE_SAMPLE=None;ACTIVE_TRACE=[]
    save_predictions(level,records)
    scores=run_longbench_eval(level)
    task_vals=[float(scores[t]) for t in TASKS]
    mean_score=round(sum(task_vals)/len(task_vals),4)
    actual=(100.*STATS['pruned_blocks']/STATS['candidate_blocks'] if STATS['candidate_blocks'] else None)
    outcome={
        'model':MODEL_ID,'gguf_file':GGUF_FILE,'level':level,'name':name,'sample_count':len(records),
        'block_size':[BLOCK_ROWS,BLOCK_COLS],'seq_len':SEQ_LEN,'tasks':TASKS,
        'proxy_definition':'P_ij=N_Q(i)*N_K(j), N=sum(INT8**2)',
        'threshold_policy':THRESHOLD_POLICY,
        'stats':dict(STATS),'measured_candidate_sparsity_percent':actual,
        'per_task_scores':scores,'average_score':mean_score,
        'threshold_trace_dir':str(level_trace_dir) if level>0 else None,
        'prediction_dir':str(pred_dir),'dataset_sources':dataset_sources,
        'elapsed_minutes_this_session':round((time.time()-start)/60,2),
    }
    atomic_json(target,outcome)
    if checkpoint_file.parent.exists():shutil.rmtree(checkpoint_file.parent)
    print('COMPLETE:',name,'LongBench score:',mean_score,'actual sparsity:',actual)
    return outcome


In [ ]:
# 12. RUN ALL LEVELS: DENSE + FOUR SPARSE
all_results={}
if RUN_BENCHMARK:
    for lev in LEVELS:
        try:
            all_results[str(lev)]=run_level_v4(lev)
            gc.collect();torch.cuda.empty_cache()
        except Exception as e:
            print('FAILED LEVEL',lev,type(e).__name__,str(e))
            print('Saved V4 checkpoints in',ROOT)
            raise
    atomic_json(RESULT_ROOT/'summary.json',all_results)
    restore_dense()
    ACTIVE_SAMPLE=None;ACTIVE_TRACE=[]
    print('All levels COMPLETE; summary:',RESULT_ROOT/'summary.json')
else:
    print('RUN_BENCHMARK=False: only deterministic golden case + self-test were executed.')


In [ ]:
# 13. SUMMARY REPORTS: T traces + per-sparsity distribution + benchmark scores
from statistics import mean,median,pstdev

def atomic_csv(path,rows,fieldnames):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=fieldnames,extrasaction='ignore');writer.writeheader();writer.writerows(rows)
    os.replace(tmp,path)

def build_reports():
    if not all_results:
        print('No benchmark levels completed; no T report generated.')
        return
    distribution=[];stats_rows=[];sparsity_rows=[];bench_rows=[]
    all_trace_count=0
    for level in LEVELS_SPARSE:
        name=level_name(level)
        merged=RESULT_ROOT/f'threshold_trace_{level}.jsonl'
        summary_rows=[]
        temp=merged.with_name(merged.name+'.tmp')
        with temp.open('w',encoding='utf-8') as out:
            for s in samples:
                source=trace_sample_path(level,s['sample_id'])
                if not source.exists():raise FileNotFoundError(source)
                with source.open(encoding='utf-8') as f:
                    for line in f:
                        if not line.strip():continue
                        rec=json.loads(line)
                        assert rec['sample_id']==s['sample_id'] and rec['target_sparsity_pct']==level
                        summary_rows.append(rec)
                        out.write(line)
        os.replace(temp,merged)
        if not summary_rows:raise AssertionError('Missing T traces: '+name)
        all_trace_count+=len(summary_rows)
        ts=[r['threshold_T'] for r in summary_rows]
        sorted_t=sorted(ts)
        def pct(p):return sorted_t[round((len(sorted_t)-1)*p)]
        protected=sum(r['diagonal_blocks'] for r in summary_rows)
        tot_candidates=sum(r['candidate_count'] for r in summary_rows)
        tot_kept=sum(r['kept_count'] for r in summary_rows)
        tot_pruned=sum(r['pruned_count'] for r in summary_rows)
        stats_rows.append({
            'target_sparsity_pct':level,'num_thresholds':len(ts),
            'mean_T':mean(ts),'median_T':median(ts),'min_T':min(ts),'max_T':max(ts),
            'std_T':pstdev(ts),'p05_T':pct(.05),'p95_T':pct(.95),
            'min_topk_T':min(r['topk_T'] for r in summary_rows),
            'max_topk_T':max(r['topk_T'] for r in summary_rows),
            'protected_diagonal_blocks':protected,
        })
        # Log10 bins enable compact meaningful distribution over 64-bit integers.
        hist={}
        for t in ts:
            bin_key=int(math.floor(math.log10(t))) if t>0 else -1
            hist[bin_key]=hist.get(bin_key,0)+1
        for bin_key, n in sorted(hist.items()):
            distribution.append({'target_sparsity_pct':level,'log10_floor_T':bin_key,'count':n,'fraction':n/len(ts)})
        sparsity_rows.append({
            'target_sparsity_pct':level,'num_thresholds':len(ts),'candidate_count':tot_candidates,
            'kept_count':tot_kept,'pruned_count':tot_pruned,
            'actual_sparsity_pct':100*tot_pruned/tot_candidates,
            'mean_per_call_sparsity_pct':mean(r['actual_sparsity_pct'] for r in summary_rows),
            'protected_diagonal_blocks':protected,
        })
    for lev in LEVELS:
        rr=all_results[str(lev)]
        for task in TASKS:
            bench_rows.append({'level':lev,'task':task,'score':rr['per_task_scores'][task],
                               'average_score':rr['average_score'],
                               'measured_candidate_sparsity_percent':rr['measured_candidate_sparsity_percent']})
    atomic_csv(RESULT_ROOT/'threshold_summary.csv',stats_rows,list(stats_rows[0]))
    atomic_csv(RESULT_ROOT/'threshold_distribution.csv',distribution,list(distribution[0]))
    atomic_csv(RESULT_ROOT/'sparsity_summary.csv',sparsity_rows,list(sparsity_rows[0]))
    atomic_csv(RESULT_ROOT/'longbench_scores.csv',bench_rows,list(bench_rows[0]))
    atomic_json(RESULT_ROOT/'threshold_config.json',{
        'norm':'N_Q=sum(q_int8**2) per 32-token/head block, N_K similarly',
        'proxy':'P=N_Q*N_K in int64, exported uint64 (RTL 64-bit)',
        'selection':'P>=T for causal valid blocks',
        'threshold_policy':THRESHOLD_POLICY,
        'tie_handling':'keep ALL proxy==T; actual sparsity may be lower than target',
        'diagonal_safety':'protect diag: threshold input per pair is 0 for q_id==k_id, else T; v3 compatible',
        'sparsities':LEVELS_SPARSE,'attention_input':'Q and repeated K after RoPE',
        'quantization':'symmetric dynamic per-head INT8 with torch.round',
        'note':'threshold_T is off-diagonal T. Actual 64-bit threshold per pair is in threshold_inputs_XX.hex',
    })
    atomic_json(RESULT_ROOT/'execution_stats.json',{str(k):v['stats'] for k,v in all_results.items()})
    print('\nTHRESHOLD SUMMARY (exact integer T; CSV mean/std are descriptive floats):')
    for r in stats_rows:
        print(f"level={r['target_sparsity_pct']}% records={r['num_thresholds']} mean_T={r['mean_T']:.4g} median_T={r['median_T']} min={r['min_T']} max={r['max_T']} protected_diagonal={r['protected_diagonal_blocks']}")
    print('TOTAL threshold records:',all_trace_count)
    print('T trace and reports saved:',RESULT_ROOT)

if RUN_BENCHMARK:build_reports()


In [ ]:
# 14. FINAL TB MANIFEST + OUTPUT AUDIT
case_dirs=sorted([d for d in TB_ROOT.iterdir() if d.is_dir() and (d/'metadata.json').exists()])
manifest={'experiment_dir':str(ROOT),'model':MODEL_ID,'gguf_file':GGUF_FILE,
  'norm_rtl_module':'Norm_Engine (source not provided)','top_module':'Norm_Base_Top',
  'reference_stage':'Dense Qwen prefill Q/K after RoPE and after GQA repeat',
  'threshold_policy':THRESHOLD_POLICY,'bus_bits':256,'norm_bits':32,'proxy_threshold_bits':64,
  'cases':[str(d.relative_to(TB_ROOT)) for d in case_dirs],
  'missing_source':'Norm_Engine.sv; full Vivado simulation not performed by this notebook',
  'usage':'Read q_int8.hex / k_int8.hex using 256-bit $readmemh, one block per Norm_Engine start; compare norm_sq, expected_proxy.hex, T, expected_mask_XX.hex; pairs.hex enumerates causal candidate IDs.'}
atomic_json(TB_ROOT/'manifest.json',manifest)
if RUN_BENCHMARK and EXPORT_TB_DATA:
    expected_cases={f'test_{sid}_L{layer:02d}_H{head:02d}_B0'
       for sid in TB_SAMPLE_IDS for layer in TB_LAYER_INDICES for head in TB_HEAD_INDICES}
    existing_cases={d.name for d in case_dirs}
    missing_cases=sorted(expected_cases-existing_cases)
    if missing_cases:
        raise RuntimeError('MISSING REAL QWEN TB ACTIVATION CASES: '+repr(missing_cases))
    print('REAL QWEN TB CASES:',len(expected_cases),'/',len(expected_cases),'PASS')
for d in case_dirs:
    m=json.loads((d/'metadata.json').read_text())
    nq,nk=m['q_blocks'],m['k_blocks'];candidate=sum(min(i+1,nk) for i in range(nq))
    assert len((d/'pairs.hex').read_text().splitlines())==candidate
    assert len((d/'expected_proxy.hex').read_text().splitlines())==candidate
    assert len((d/'q_int8.hex').read_text().splitlines())==nq*m['q_beats_per_block']
    assert len((d/'k_int8.hex').read_text().splitlines())==nk*m['k_beats_per_block']
    for lev in LEVELS_SPARSE:
        t=int((d/f'threshold_{lev}.hex').read_text().strip(),16)
        pp=[int(s,16) for s in (d/'expected_proxy.hex').read_text().splitlines()]
        tt=[int(s,16) for s in (d/f'threshold_inputs_{lev}.hex').read_text().splitlines()]
        mask=[int(s,16) for s in (d/f'expected_mask_{lev}.hex').read_text().splitlines()]
        assert all(x in (0,t) for x in tt)
        assert mask==[int(x>=y) for x,y in zip(pp,tt)],f'RTL comparator mismatch {d} lev={lev}'
print('GGUF valid:',check_gguf(gguf_path),'| cases:',len(case_dirs),'| Vivado manifest:',TB_ROOT/'manifest.json')
if RUN_BENCHMARK:
    required=['summary.json','threshold_summary.csv','threshold_distribution.csv',
        'threshold_config.json','sparsity_summary.csv','longbench_scores.csv','execution_stats.json']
    required += [f'threshold_trace_{level}.jsonl' for level in LEVELS_SPARSE]
    missing=[p for p in required if not (RESULT_ROOT/p).exists()]
    if missing:raise RuntimeError('MISSING REQUIRED V4 OUTPUTS: '+repr(missing))
    for level in LEVELS:
        name=level_name(level)
        assert (RESULT_ROOT/f'{name}.json').exists()
        assert all((PRED_ROOT/name/f'{t}.jsonl').exists() for t in TASKS)
    print('ALL REQUIRED BENCHMARK/T OUTPUT FILES: PASS')
else:
    print('Benchmark disabled: not claiming benchmark/T trace completed.')
print('RESULT DIRECTORY:',RESULT_ROOT)
print('TB DATA DIRECTORY:',TB_ROOT)
restore_dense()


## Hướng dẫn nối với Vivado (không thay đổi HDL gốc)

1. `q_int8.hex`, `k_int8.hex`: khai báo `logic [255:0] q_mem [0:...];` rồi `$readmemh(...)`; mỗi **block ID** nạp `q_beats_per_block` beat liên tiếp và đánh `data_last` ở beat cuối, **chỉ sau khi `q_start`/`k_start` đã được chấp nhận**. Kích thước 256 bit xuất phát từ `q_data/k_data` trong `Norm_Base_Top.sv`.
2. So sánh `Norm_Engine.norm_sq`/`decision_q_norm`/`decision_k_norm` với `q_norm_sq.hex`, `k_norm_sq.hex` theo đúng ID. Lưu ý `Norm_RAM` có synchronous read và entry valid, nên phải chờ các block đã nạp hoàn tất trước khi gửi pair.
3. Cho `pair_q_id`/`pair_k_id` đi theo `pairs.hex`. Với từng tỷ lệ, nạp **từng dòng** `threshold_inputs_XX.hex` vào `threshold_T` **tại lúc pair handshake**; đây là 0 nếu diagonal, T (`threshold_XX.hex`) nếu off-diagonal. Nếu giữ T cố định cho tất cả cặp, RTL sẽ **không** bảo vệ diagonal và mask sẽ không khớp. So `decision_score` với `expected_proxy.hex`, `decision_keep` với `expected_mask_XX.hex`. Ghi nhận `decision_valid`/độ trễ.
4. `pairs.hex` liệt kê **chỉ các block hợp lệ causal**. Để kiểm tra `Causal_Block_Mask` riêng, gửi thêm các cặp `k_block_id>q_block_id`, mong `causal_prune=1` và `out_valid=0`.
5. **Chưa thể chạy full RTL top** vì thiếu `Norm_Engine.sv`; phần notebook chỉ chứng minh golden model và mối quan hệ comparator, **không tự tuyên bố RTL PASS**.
6. Kết quả ngưỡng để viết paper/report: `results/threshold_summary.csv` và **full trace** `results/threshold_trace_{50,70,80,90}.jsonl`. Để nạp phần cứng dùng **exact uint64 per pair** từ `vivado_testdata/test_.../threshold_inputs_XX.hex`; `threshold_XX.hex` là T ngoài đường chéo, không dùng mean làm tròn trong CSV.
